In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from torchvision import transforms, datasets

preprocessing = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

])

train = datasets.ImageFolder(root= '/content/drive/MyDrive/data/train', transform= preprocessing)
test = datasets.ImageFolder(root= '/content/drive/MyDrive/data/train', transform= preprocessing)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train, batch_size= 10, shuffle= True, num_workers= 6)
test_loader = DataLoader(test, batch_size= 10, num_workers= 6)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


torch.Size([10, 3, 28, 28])
torch.Size([10])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomCNN(nn.Module):
    def __init__(self, num_classes = 3):
        super(CustomCNN, self).__init__()

        # First Convolutional Layer
        # in-> (3, 28, 28)
        self.conv1 = nn.Conv2d(in_channels= 3, out_channels= 16, kernel_size= 3, padding= 1)

        # Second Convolutional Layer
        # in-> (16, 14, 14)
        self.conv2 = nn.Conv2d(in_channels= 16, out_channels= 32, kernel_size= 3, padding= 1)

        # Pooling
        self.pooling= nn.MaxPool2d(2, 2)

        # Dropout
        self.dropout = nn.Dropout(0.6)

        # First Fully Connected Layer
        # in-> (32, 7, 7)
        self.fc1 = nn.Linear(32*7*7, 128)


        # Second Fully Connected Layer (output Layer)
        # in-> (256)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):

        # conv1
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pooling(x)

        # conv2
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pooling(x)

        # Flatten
        x = torch.flatten(x, start_dim= 1)
        # fc1
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)

        # fc2 (Output Layer)
        x = self.fc2(x)

        return x

In [ ]:
# set to cuda

device = 'cuda'

# model (object from the class 'CustomCNN')
model = CustomCNN(3).to(device)

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr= 0.001, weight_decay= 0.0005)    # weight_decay param is R2 Norm (Ridge Regularization)

In [ ]:
# train


epochs = 8

for epoch in range(epochs):
    model.train()
    training_loss = 0
    total = 0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)    # It returns the value and the position here sets the value for var '_', and the position (that we need) to the var 'predicted'

        training_loss += loss.item() * labels.size(0)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_epoch_loss = training_loss / total
    avg_epoch_acc = correct /total

    print(f"Epoch {epoch + 1} | Loss : {avg_epoch_loss:.4f}, Acc : {avg_epoch_acc:.4f}")

Epoch 1 | Loss : 0.7940, Acc : 0.5813
Epoch 2 | Loss : 0.5317, Acc : 0.7396
Epoch 3 | Loss : 0.3907, Acc : 0.8208
Epoch 4 | Loss : 0.2473, Acc : 0.9042
Epoch 5 | Loss : 0.1396, Acc : 0.9500
Epoch 6 | Loss : 0.0650, Acc : 0.9812
Epoch 7 | Loss : 0.0386, Acc : 0.9917
Epoch 8 | Loss : 0.0289, Acc : 0.9917


In [ ]:
# test

model.eval()

testing_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        _, predicted = torch.max(outputs, 1)

        testing_loss += loss.item() * labels.size(0)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    print(f"Loss : {testing_loss / total : .4f}, Acc : {correct / total : .4f}")

Loss :  0.0049, Acc :  1.0000
